In [12]:
from evaluate_model import evaluate
import pandas as pd
import joblib
from sklearn.metrics import precision_recall_curve
from preprocessing import split_scale
import numpy as np

In [6]:
def get_performance_table():
    df_path='data/processed/processed_creditcard.csv'

    models_to_evaluate=[
        ('models/logistic_model.pkl','Logistic Regression'),
        ('models/randomforest_model.pkl','RandomForest (SMOTE)'),
        ('models/randomforest_nosmote_model.pkl','RandomForest (no SMOTE)')
    ]

    results=[]
    for model_path, name in models_to_evaluate:
        try:
            results.append(evaluate(model_path,df_path,name))
        except Exception as e:
            print(f'Error with {name}:{e}')

    performance_table=pd.DataFrame(results)
    cols=['Model','ROC-AUC','PR-AUC','Accuracy','Precision (Fraud=1)', 'Recall (Fraud=1)','F1-Score (Fraud=1)','TP','FP','FN','TN']
    performance_table=performance_table[cols]

    return performance_table


In [7]:
table=get_performance_table()

print('Model Performance Comparison')

display(table.round(4))

Model Performance Comparison


,Model,ROC-AUC,PR-AUC,Accuracy,Precision (Fraud=1),Recall (Fraud=1),F1-Score (Fraud=1),TP,FP,FN,TN
0,Logistic Regression,0.9715,0.7215,0.9734,0.0558,0.9082,0.1051,89,1506,9,55358
1,RandomForest (SMOTE),0.9824,0.8740,0.9995,0.8901,0.8265,0.8571,81,10,17,56854
2,RandomForest (no SMOTE),0.9626,0.8749,0.9996,0.9512,0.7959,0.8667,78,4,20,56860


In [15]:
### Finding the best threshold

def find_best_threshold(model_path, df_path,target='f1'):
    df=pd.read_csv(df_path)
    model= joblib.load(model_path)
    _,X_test,_,y_test,_=split_scale(df)

    probs=model.predict_proba(X_test)[:,1]
    precision, recall, thresholds= precision_recall_curve(y_test, probs)

    if target=='f1':
        scores=2 * (precision * recall)/(precision + recall + 1e-9)
    else:
        scores=recall

    best_idx=np.argmax(scores)

    best_threshold=thresholds[best_idx]

    return{
        'best_threshold':round(float(best_threshold),4),
        'best_score': round(float(scores[best_idx]),4),
        'precision': round(float(precision[best_idx]),4),
        'recall':round(float(recall[best_idx]),4)
    }


In [16]:
tuned_results=[]

models_list=[
        ('models/logistic_model.pkl','Logistic Regression'),
        ('models/randomforest_model.pkl','RandomForest (SMOTE)'),
        ('models/randomforest_nosmote_model.pkl','RandomForest (no SMOTE)')
    ]

for model, name in models_list:
    try:
        result=find_best_threshold(model, 'data/processed/processed_creditcard.csv')
        result['Model']=name
        tuned_results.append(result)

    except Exception as e:
        print(f'Error tuninh {name}:{e}')

tuned_table=pd.DataFrame(tuned_results)
print('\nTuned Threshold Results')
display(tuned_table.round(4))



Tuned Threshold Results


,best_threshold,best_score,precision,recall,Model
0,1.000,0.8247,0.8333,0.8163,Logistic Regression
1,0.545,0.8649,0.9195,0.8163,RandomForest (SMOTE)
2,0.450,0.8865,0.9425,0.8367,RandomForest (no SMOTE)


In [17]:
final_results = []

for model_path, name in models_list:
    try:
        # Get best threshold for this model
        best = find_best_threshold(model_path, 'data/processed/processed_creditcard.csv')
        threshold = best['best_threshold']
        
        # Evaluate at tuned threshold
        df = pd.read_csv('data/processed/processed_creditcard.csv')
        model = joblib.load(model_path)
        _, X_test, _, y_test, _ = split_scale(df)
        
        probs = model.predict_proba(X_test)[:, 1]
        y_pred_tuned = (probs >= threshold).astype(int)
        
        from sklearn.metrics import classification_report, confusion_matrix
        report = classification_report(y_test, y_pred_tuned, output_dict=True, zero_division=0)
        cm = confusion_matrix(y_test, y_pred_tuned)
        tn, fp, fn, tp = cm.ravel()
        
        final_results.append({
            'Model': f"{name} (tuned)",
            'Threshold': threshold,
            'Precision (Fraud=1)': report['1']['precision'],
            'Recall (Fraud=1)': report['1']['recall'],
            'F1-Score (Fraud=1)': report['1']['f1-score'],
            'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn)
        })
    except Exception as e:
        print(f"Error with {name}: {e}")

final_table = pd.DataFrame(final_results)
display(final_table.round(4))

,Model,Threshold,Precision (Fraud=1),Recall (Fraud=1),F1-Score (Fraud=1),TP,FP,FN,TN
0,Logistic Regression (tuned),1.000,0.8167,0.5000,0.6203,49,11,49,56853
1,RandomForest (SMOTE) (tuned),0.545,0.9195,0.8163,0.8649,80,7,18,56857
2,RandomForest (no SMOTE) (tuned),0.450,0.9425,0.8367,0.8865,82,5,16,56859
